<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/22_fbeta_and_transaction_costs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, fbeta_score
from src.experiments.helpers import TICKERS, build_event_data, align_columns
from src.backtest.simulator import simulate_trades

event_data = {t: build_event_data(t) for t in TICKERS}
cols = align_columns(event_data)

def trade_class_fbeta(y, pred, beta):
    # F-beta on long/short only: his argument is about the cost of opening positions
    return fbeta_score(y, pred, labels=['long', 'short'], beta=beta, average='macro', zero_division=0)

store = {}
for holdout in ['XOM', 'JPM']:
    train_t = [t for t in TICKERS if t != holdout]
    X = pd.concat([event_data[t]['features'] for t in train_t], ignore_index=True).fillna(0)
    y = pd.concat([event_data[t]['labels'] for t in train_t], ignore_index=True)
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced',
                                random_state=42, n_jobs=-1).fit(X, y)
    Xh = event_data[holdout]['features'].fillna(0)
    yh = event_data[holdout]['labels']
    pred = rf.predict(Xh)
    proba = pd.DataFrame(rf.predict_proba(Xh), columns=rf.classes_)
    store[holdout] = {'y': yh, 'pred': pred, 'proba': proba,
                      'labeled_df': event_data[holdout]['labeled_df'],
                      'raw': event_data[holdout]['raw']}
    print(f"{holdout}: F1 macro={f1_score(yh, pred, average='macro'):.3f}, "
          f"F0.5 (long/short)={trade_class_fbeta(yh, pred, 0.5):.3f}, "
          f"F1 (long/short)={trade_class_fbeta(yh, pred, 1.0):.3f}, "
          f"F2 (long/short)={trade_class_fbeta(yh, pred, 2.0):.3f}")